### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="kickstarter",
    dataset_year="2025",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://webrobots.io/kickstarter-datasets/",
    download_description="""
There exists data from Kaggle (https://www.kaggle.com/datasets/yashkantharia/kickstarter-campaign).
We use data from the original source (https://webrobots.io/kickstarter-datasets/) and downloaded the newest version at the time of writing.

wget https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip
mkdir -p local-data-warehouse/kickstarter && mv Kickstarter_2026-01-12T09_37_51_016Z.zip local-data-warehouse/kickstarter && mkdir -p local-data-warehouse/kickstarter/data_files && unzip local-data-warehouse/kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip -d local-data-warehouse/kickstarter/data_files
""",
    # References
    academic_reference_bibtex=r"""@misc{webrobots2026kickstarter,
  title        = {Kickstarter Datasets},
  author       = {{Web Robots}},
  howpublished = {\url{https://webrobots.io/kickstarter-datasets/}},
  note         = {Accessed: 2026-01-25},
  year         = {2026},
  organization = {Web Robots}
}
""",
    academic_reference_bibtex_key="webrobots2026kickstarter",
    license="None",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
Similar to the Kaggle task, we aim to predict whether a Kickstarter project will be funded successfully.

- The data we used only had 2 samples for 2026, so we stick to data until 2025.
- The original data comes in .csv files per scrape data. We first combine all files into one file.
- We remove all columns that are known only after the outcome and thus could leak information.
- The fx_rate seems to show the conversion rate at the time of crawling the data and not at the time of the project. Thus, if it is used as on Kaggle, the numbers are wrong. We use the "usd_exchange_rate" to transform the goal into USD currency.
- We drop the two entries with "disable_communication" as they point to other issues.
- We drop photo and video based references as we do not include these modalities.
- We decode the category and creator name  from JSON strings into usable columns.
- Note, the crawl does only include blurbs and not the full-text descriptions of the projects. Also some blurb repeat from similar projects or orders.
- We decode the profile blurb, when it exists.
- We decode the location display name from the location JSON string, which can include state and city name.
- The data contains spatial information (city, state, country). We do not decode this but leave it to the pipelines.
- We found 150 rows without location information. We drop these rows as it is unclear which issue caused this and how the rows' data might be affected by this.
- We drop duplicates (20%) which seems to have occurred from multiple scrapes of the same project.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="state",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="state",
    time_on="created_at", # could also be deadline to get more realistic data with projects for which we know the lab at "real" train time. But with our time splits, created_at is fine as well.
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import json

data_dir = dataset_mold.path / "data_files"

# Read and concatenate all CSV files
df = pd.concat(
    (pd.read_csv(csv_file) for csv_file in data_dir.glob("*.csv")),
    ignore_index=True
)
print("Loaded data shape:", df.shape)

# Filter to successful or failed projects
df = df[df["state"].isin(["successful", "failed"])]

# Drop rows with missing location information
df = df[~df["location"].isna()]

# Convert goal to USD
non_usd_mask = df["currency"] != "USD"
df.loc[non_usd_mask, "goal"] = df.loc[non_usd_mask, "goal"] * df.loc[non_usd_mask, "usd_exchange_rate"]

# Use only full country name
df["country"] = df["country_displayable_name"]
df = df.drop(columns=["country_displayable_name"])
# drop disable_communication artifacts
df = df[df["disable_communication"] == False]
df = df.drop(columns=["disable_communication"])

# We do not include images for this benchmark, so we drop the related columns
df = df.drop(columns=["photo", "video"])

# Drop other columns
drop_columns = [
    # Remove target leakage columns
    "backers_count", "converted_pledged_amount", "is_in_post_campaign_pledging_phase",
    "percent_funded", "pledged", "static_usd_rate", "usd_pledged", "usd_type",
    "usd_exchange_rate", "state_changed_at", "spotlight",
    # Remove currency related columns not need anymore
    "currency", "currency_symbol", "currency_trailing_code",
    "current_currency", "fx_rate",
    # all unique
    "is_disliked", "is_launched", "is_liked", "is_starrable",
    # We got the date from created_at
    "id",
    # Just formating of name
    "slug",
    # No relation / not needed
    "source_url", "urls",
]
df = df.drop(columns=drop_columns)

# Decode category
df["main_category"] = df["category"].apply(lambda x: json.loads(x)["name"])
df["sub_category"] = df["category"].apply(lambda x: json.loads(x)["parent_name"] if "parent_name" in json.loads(x) else np.nan)
df = df.drop(columns=["category"])

# Decode profile blurb
def decode_profile(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"blurb":')]
        assert len(raw_data) == 1
        blurb_entry = raw_data[0]
        blurb_entry = blurb_entry.replace('"blurb":"', "").rstrip('"')
        if "null" in blurb_entry:
            return np.nan
        if blurb_entry == "":
            return np.nan
        return blurb_entry
    res = data["blurb"]
    if res == "":
        return np.nan
    return res

df["profile_blurb"] = df["profile"].apply(decode_profile)
df = df.drop(columns=["profile"])

# Decode creator name
def decode_name(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"name":')]
        assert len(raw_data) == 1
        name_entry = raw_data[0]
        name_entry = name_entry.replace('"name":"', "").rstrip('"')
        return name_entry
    return data["name"]
df["creator_name"] = df["creator"].apply(decode_name)
df = df.drop(columns=["creator"])

# Decode location
df["location_displayable_name"] = df["location"].apply(lambda x: json.loads(x)["displayable_name"])
df = df.drop(columns=["location"])

# Dtypes
as_string_cols = [
    "blurb",
    "name",
    "creator_name",
    "profile_blurb",
    "location_displayable_name",
]
as_date_cols = [
    "created_at",
    "launched_at",
    "deadline",
]
as_cat_type = [
    "prelaunch_activated",
    "staff_pick",
    "main_category",
    "sub_category",
    "state",
    "country",
]

for c in as_string_cols:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

for c in as_date_cols:
    df[c] = pd.to_datetime(df[c], unit="s")

df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicates
df = df.drop_duplicates()

# Drop the 2 samples from 2026
df = df[df["created_at"].dt.year < 2026]

df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
# Filter to year 2025

# Create a year-month column for grouping
df["year"] = df[task_mold.time_on].dt.to_period("Y")
monthly_totals = (
    df
    .groupby("year")
    .size()
    .rename("total_samples")
)
monthly_class_counts = (
    df
    .groupby(["year", task_mold.target_column_name])
    .size()
    .unstack(fill_value=0)
)
result = monthly_class_counts.join(monthly_totals)
result = result.sort_index()
result.index = result.index.to_timestamp()
result

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_years = [
    "2023",
    "2024",
    "2025",
]
for i, month in enumerate(test_years):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated monthly but has not enough data per month to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set.

 We could simulate a model that is refitted every month, but this would need many splits. Moreover, data from just one month is not enough to create a robust test set. We instead simulate a model that is refit every year. This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model. We create 3 test splits by 2023, 2024, and 2025 as test year. For each test split, we use all data before the test month as training data.
""",
    splits=splits,
    time_horizon=1,
    time_horizon_unit="years",
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()